# Session 4 Lab: Chat with the College FAQ (RAG)

**Project 3 · Chat with the syllabus** · GenAI & Agentic AI Workshop

You'll build a RAG chatbot step by step: **load → chunk → embed → store → retrieve → answer with sources**.

We use a sample FAQ for a fictional "ABC Engineering College". At the end you can upload your own college's PDF.

**Before you start**
1. You need the same `GEMINI_API_KEY` secret as in Session 3 (🔑 Secrets, Notebook access ON).
2. Run each cell with **Shift + Enter**. The first install takes about a minute.

In [ ]:
!pip install -q google-genai chromadb sentence-transformers pypdf

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"   # any current free model from Google AI Studio works

## Step 0: Why do we need RAG?

Ask the model about our college **without** giving it any documents. It has never seen this FAQ, so watch what it does.

In [ ]:
q = "What is the revaluation fee at ABC Engineering College, and how many days do I have to apply?"
print(client.models.generate_content(model=MODEL, contents=q).text)

It either guesses (hallucination) or says it doesn't know. Either way, it can't help. Let's fix that.

## Step 1: Load the document

This cell saves the sample FAQ as a file, like a document you'd download from your college website.

In [ ]:
%%writefile abc-college-faq.md
# ABC Engineering College: Student FAQ

This is a sample document for the workshop. ABC Engineering College is a fictional college. Use your own college's documents for your project.

## Exams

Semester-end exams are held in December (odd semesters) and May (even semesters). Each subject has three internal assessments per semester, and the best two of the three are counted. Hall tickets can be downloaded from the student portal one week before the exams start. You will only get a hall ticket if you have at least 75% attendance in that subject. Results are published on the student portal about six weeks after the last exam. To apply for revaluation, log in to the student portal within 7 days of the result and pay a fee of ₹500 per subject. Revaluation results come out within 30 days.

## Attendance

You need at least 75% attendance in every subject. If your attendance is between 65% and 74%, you can apply for condonation with a medical certificate and a fee of ₹1,000; the principal decides each case. Students below 65% in a subject are detained and must repeat that subject. Attendance is updated on the student portal every Friday. If your attendance is marked wrongly, report it to the subject teacher within 7 days.

## Fees

Tuition fees are paid in two instalments: the first by 15 July and the second by 15 January. You can pay online through the student portal using UPI, net banking or a debit card, or with a bank challan at the campus SBI branch. A late fee of ₹100 per day is charged after the due date, up to a maximum of ₹2,000. Fee receipts can be downloaded from the portal under My Payments. For fee questions, visit the accounts office in Admin Block, Room 102.

## Hostel

The college has separate hostels for boys and girls. Apply for a hostel room in June through the student portal; rooms are given on a first-come, first-served basis. The hostel fee is ₹85,000 per year, including mess charges. The hostel in-time is 9:30 PM. Visitors are allowed on Sundays from 10 AM to 5 PM in the visitors' room only. Hostel WiFi or maintenance problems should be raised as a complaint on the portal under Hostel Help; the warden responds within 48 hours.

## Library

The central library is open Monday to Saturday from 8 AM to 8 PM. It is closed on Sundays, except during exam months (December and May), when it opens on Sundays from 10 AM to 4 PM. Students can borrow up to 4 books for 14 days. The late return fine is ₹2 per book per day. The digital library gives free access to IEEE papers, NPTEL courses and e-books through your college login, on or off campus.

## Placements

The Training and Placement (T&P) cell is in Block A, second floor. Students must register on the placement portal in their 6th semester, before 31 July. To be eligible for most companies, you need a CGPA of at least 6.5 and no active backlogs, though some companies set their own criteria. The college follows a one-student-one-job policy: once you accept an offer of ₹6 LPA or more, you cannot sit for further placement drives. Pre-placement training (aptitude, coding and mock interviews) runs every Saturday during the 6th semester. An 8-week internship after the 6th semester is mandatory for all B.Tech students.

## Scholarships

The college gives a merit scholarship of a 25% tuition fee waiver to the top 5% of students in each branch who have a CGPA of 9.0 or above. State government scholarships are applied for through the state scholarship (SSP) portal; the deadline is 31 October each year. For help with scholarship forms, visit the scholarship help desk in the Admin Block, Room 104, Monday to Friday, 10 AM to 1 PM.

## Clubs and events

Student clubs include CodeCraft (coding), RoboMinds (robotics), NSS (social service), and Rhythm (music and dance). Clubs recruit new members in the first two weeks of the odd semester. The college tech fest, TechNova, is held every February and includes a 24-hour hackathon and a robo-race. The cultural fest, Utsav, is held in April.

## Contact

The college office is open Monday to Saturday, 9:30 AM to 4:30 PM. Email office@abc.edu or call the front desk at 080-1234-5678. For urgent hostel issues at night, call the hostel warden's helpline listed on the hostel notice board.

In [ ]:
text = open("abc-college-faq.md", encoding="utf-8").read()
print(len(text.split()), "words")
print(text[:500])

## Step 2: Chunk it

We cut the document into small, overlapping pieces. First a simple word-based chunker:

In [ ]:
def chunk_text(text, size=120, overlap=20):
    """Split text into chunks of `size` words, each overlapping the previous by `overlap` words."""
    words = text.split()
    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks

demo = chunk_text(text, size=60, overlap=10)
print(len(demo), "chunks")
print("--- chunk 1 ---\n", demo[0])
print("--- chunk 2 ---\n", demo[1])

Notice the end of chunk 1 repeats at the start of chunk 2: that's the **overlap**.

Our FAQ has clear `## headings`, so a smarter way is to split **by section first**, then chunk long sections. The heading becomes the **source** we cite later.

In [ ]:
def split_by_heading(md_text, size=120, overlap=20):
    chunks = []
    for block in md_text.split("\n## ")[1:]:          # skip the title/intro
        title, _, body = block.partition("\n")
        for i, piece in enumerate(chunk_text(body, size, overlap)):
            chunks.append({"id": f"{title.strip()}-{i}", "source": title.strip(), "text": piece})
    return chunks

chunks = split_by_heading(text)
print(len(chunks), "chunks")
for c in chunks:
    print(f"[{c['source']}] {c['text'][:70]}...")

## Step 3: Embed the chunks

Turn every chunk into a vector (Session 1). We use a small free model that runs inside Colab, so there are no API limits for this step.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(doc_vecs.shape, "→ one vector of", doc_vecs.shape[1], "numbers per chunk")

## Step 4: Search by meaning (RAG from scratch, no database)

To search, embed the question with the **same model** and find the chunks with the highest similarity.

In [ ]:
import numpy as np

def search(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    scores = doc_vecs @ q                    # cosine similarity (vectors are normalized)
    top = np.argsort(-scores)[:k]
    return [(chunks[i]["source"], round(float(scores[i]), 2)) for i in top]

for question in ["Is the library open on Sunday?",
                 "How much is the late fee if I pay tuition after the deadline?",
                 "Can I keep attending placement drives after getting a 7 LPA offer?",
                 "Who won the IPL in 2020?"]:
    print(question, "→", search(question))

Look at the scores. Relevant questions score high on the right section. The IPL question scores low everywhere: nothing in the FAQ matches.

## Step 5: Store in a vector database (Chroma)

Our NumPy search works for 20 chunks. For thousands of documents we use a **vector database**, which also stores the text and source with each vector.

In [ ]:
import chromadb

db = chromadb.Client()                        # in-memory; use chromadb.PersistentClient(path="db") to save to disk
try:
    db.delete_collection("college_faq")       # start fresh if you re-run this cell
except Exception:
    pass
col = db.create_collection("college_faq")

col.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
    embeddings=doc_vecs.tolist(),
)
print(col.count(), "chunks stored")

In [ ]:
def retrieve(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    hits = []
    for text, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        score = 1 - dist / 2                  # Chroma returns squared L2 distance; for normalized vectors this gives cosine similarity
        hits.append({"source": meta["source"], "text": text, "score": round(score, 2)})
    return hits

for h in retrieve("How do I apply for a hostel room?"):
    print(h["score"], h["source"], "|", h["text"][:80], "...")

## Step 6: Generate an answer with sources

Now the "G" in RAG. We put the retrieved chunks into the prompt, and tell the model to answer **only** from them and cite the source.

In [ ]:
SYSTEM = """You are CampusBuddy, the helpdesk assistant for ABC Engineering College.
Answer the student's question using ONLY the context below.
- After each fact, cite the source in square brackets, like [Exams].
- If the answer is not in the context, say: "I couldn't find that in the college documents. Please contact office@abc.edu."
- Keep the answer under 80 words, in simple English."""

MIN_SCORE = 0.3   # below this, we treat a chunk as "not relevant". Try changing it!

def ask(question, k=3, show_sources=True):
    hits = [h for h in retrieve(question, k) if h["score"] >= MIN_SCORE]
    if not hits:
        return "I couldn't find that in the college documents. Please contact office@abc.edu."
    context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0),
    )
    if show_sources:
        print("   (retrieved:", ", ".join(f"{h['source']} {h['score']}" for h in hits), ")")
    return resp.text

print(ask("What is the revaluation fee, and how many days do I have to apply?"))

Compare this with Step 0. Same model, but now it has the right page in front of it.

## Step 7: Test it

Try questions that **are** in the FAQ, and questions that **aren't**. A good RAG bot answers the first kind with a source and politely refuses the second kind.

In [ ]:
tests = [
    "Is the library open on Sunday?",
    "My attendance in DBMS is 70%. Can I still write the exam?",
    "When is the second fee instalment due, and what if I'm late?",
    "What CGPA do I need for placements?",
    "Who is the principal of the college?",      # not in the FAQ
    "What is the capital of France?",            # not about the college
]
for t in tests:
    print("Q:", t)
    print("A:", ask(t), "\n")

## Step 8: Evaluate retrieval

Before blaming the LLM, check that **retrieval** found the right chunk. Write questions with the section you expect, and count how often it's in the top 3.

In [ ]:
eval_set = [
    ("How many books can I borrow?", "Library"),
    ("What is the hostel fee per year?", "Hostel"),
    ("When are hall tickets available?", "Exams"),
    ("What is the one-student-one-job rule?", "Placements"),
    ("Deadline for the state scholarship?", "Scholarships"),
    ("When is the tech fest?", "Clubs and events"),
    ("What happens if my attendance is 60%?", "Attendance"),
    ("How do I pay my fees online?", "Fees"),
]
hits = 0
for q, expected in eval_set:
    found = [h["source"] for h in retrieve(q, k=3)]
    ok = expected in found
    hits += ok
    print("✅" if ok else "❌", q, "→", found)
print(f"\nRetrieval hit rate: {hits}/{len(eval_set)}")

## Step 9: Chat with it

Type your questions. Type `quit` to stop.

In [ ]:
while True:
    q = input("You: ").strip()
    if q.lower() in ("quit", "exit", "bye", ""):
        break
    print("CampusBuddy:", ask(q), "\n")

## Stretch 1: Use your own college's PDF

Upload a syllabus, placement brochure or FAQ PDF. Each chunk's source will be the file name and page number.

In [ ]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()                     # choose a PDF
name = next(iter(uploaded))
reader = PdfReader(name)

pdf_chunks = []
for page_no, page in enumerate(reader.pages, start=1):
    for i, piece in enumerate(chunk_text(page.extract_text() or "", size=120, overlap=20)):
        pdf_chunks.append({"id": f"{name}-p{page_no}-{i}", "source": f"{name} p.{page_no}", "text": piece})

pdf_vecs = embedder.encode([c["text"] for c in pdf_chunks], normalize_embeddings=True)
col.add(
    ids=[c["id"] for c in pdf_chunks],
    documents=[c["text"] for c in pdf_chunks],
    metadatas=[{"source": c["source"]} for c in pdf_chunks],
    embeddings=pdf_vecs.tolist(),
)
print(f"Added {len(pdf_chunks)} chunks from {len(reader.pages)} pages. Total: {col.count()}")

Now go back to Step 9 and ask questions about your PDF. Scanned PDFs (images of text) won't work with `pypdf`; they need OCR.

## Stretch 2: Experiments

- Change `size` in `split_by_heading` to 40 and to 400, re-run Steps 2 to 8. What happens to the hit rate and the answers?
- Change `k` from 3 to 1 and to 6.
- Change `MIN_SCORE` to 0.1 and to 0.6. Which questions break?

## Stretch 3: Add memory

Combine this with Session 3: keep a `history` list so follow-up questions like *"and what about Sundays?"* work. Hint: retrieve using the last two user messages joined together.

## Done? 🎉

Save a copy (**File → Save a copy in Drive**) and push it to your GitHub as Project 3. Tomorrow we turn CampusBuddy into an **agent** that can also call tools.